# W9C2 Lab: Teaching a Model What People Prefer

Run every cell from the top. **Everything already works.**

Today you will:

1. Turn pairwise human preferences into a reward model.
2. Watch it learn to score answers nobody labelled.
3. See it get gamed by the thing it was not told to measure.

There is no test to run and nothing to submit. Each task tells you what
you should see when it is right.

In [ ]:
# Setup. No LLM needed: the answers and the preferences are given.
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer

torch.manual_seed(0); np.random.seed(0)

# Twelve candidate answers to "how do I reset my password?".
ANSWERS = [
    "Go to Settings, choose Security, then click Reset Password.",
    "Open Settings, select Security, and use the Reset Password link.",
    "In Settings under Security you will find a Reset Password option.",
    "Click Reset Password on the Security page inside Settings.",
    "Just google it.",
    "idk try turning it off and on",
    "You cannot reset it.",
    "Read the manual.",
    "Password. Reset. Settings. Security. Click.",
    "RESET PASSWORD SETTINGS SECURITY NOW",
    "To reset your password, open Settings and look under Security.",
    "no",
]
HELPFUL = [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0]     # what a human actually thinks

print(f"{len(ANSWERS)} candidate answers, {sum(HELPFUL)} of them helpful")

## Part 1. People compare, they do not score

Nobody can reliably say 'that answer is a 7 out of 10'. But anyone can say
'this one is better than that one'. RLHF is built on that fact.

In [ ]:
# GIVEN. Build pairwise preferences from the labels.
pairs = [(i, j) for i in range(len(ANSWERS)) for j in range(len(ANSWERS))
         if HELPFUL[i] == 1 and HELPFUL[j] == 0]

print(f"{len(pairs)} preference pairs, each saying 'left is better than right'")
for i, j in pairs[:4]:
    print(f"   PREFER: {ANSWERS[i][:44]:<46} OVER: {ANSWERS[j][:30]}")

In [ ]:
# GIVEN. The reward model, and the loss that makes a comparison trainable.
vectorizer = CountVectorizer()
features = torch.tensor(vectorizer.fit_transform(ANSWERS).toarray(), dtype=torch.float32)

reward_model = nn.Linear(features.shape[1], 1)
opt = torch.optim.Adam(reward_model.parameters(), lr=0.05)

losses = []
for step in range(300):
    scores = reward_model(features).squeeze(1)
    better = scores[[i for i, j in pairs]]
    worse = scores[[j for i, j in pairs]]
    # Bradley-Terry: make the preferred answer score higher, by any margin.
    loss = -torch.log(torch.sigmoid(better - worse)).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())

plt.figure(figsize=(6, 2.8))
plt.plot(losses, color="#7C2529"); plt.xlabel("step"); plt.ylabel("preference loss")
plt.title("Learning what people prefer"); plt.show()

with torch.no_grad():
    final = reward_model(features).squeeze(1).numpy()
out = pd.DataFrame({"reward": final.round(2), "helpful": HELPFUL,
                    "answer": [a[:46] for a in ANSWERS]}).sort_values("reward", ascending=False)
print(out.to_string(index=False))

In [ ]:
# ================== YOUR TURN 1 ==================
# Write a NEW answer the model has never seen and score it.
#
# Try a genuinely good one, then a bad one.
#
# Expected: good answers score positive, bad ones negative, even though nobody
#           labelled yours. That generalisation is the whole point: you label a few
#           thousand comparisons and get a scorer for everything.
# ===============================================
MY_ANSWER = "Open Settings, go to Security, and choose Reset Password."   # <-- change me

v = torch.tensor(vectorizer.transform([MY_ANSWER]).toarray(), dtype=torch.float32)
with torch.no_grad():
    print(f"reward: {reward_model(v).item():+.2f}")
print("(positive means the model thinks a human would prefer it)")

## Part 2. Now game it

A reward model scores text, and text is easy to manipulate. This is the
failure mode the KL penalty in RLHF exists to prevent.

In [ ]:
# GIVEN. What did the model actually reward?
weights = reward_model.weight.detach().numpy()[0]
vocab = vectorizer.get_feature_names_out()
order = np.argsort(weights)

print("words that RAISE the reward:", [vocab[i] for i in order[-6:]])
print("words that LOWER it:        ", [vocab[i] for i in order[:6]])

plt.figure(figsize=(8, 3))
top = np.concatenate([order[:6], order[-6:]])
plt.bar([vocab[i] for i in top], weights[top], color=["#999"] * 6 + ["#7C2529"] * 6)
plt.axhline(0, color="black", linewidth=0.8); plt.xticks(rotation=45, ha="right")
plt.ylabel("effect on reward"); plt.tight_layout(); plt.show()

In [ ]:
# ================== YOUR TURN 2 ==================
# Build a nonsense answer out of the high-reward words and see whether
# the model rewards it anyway.
#
# Start from the list printed above.
#
# Expected: at REPEATS = 1 the nonsense string roughly ties the real answer
#           (+8.7 against +8.8). At 2 it scores +17.3, nearly double, and at 5 it
#           reaches +42.9. Saying the same six words over and over beats a genuine
#           answer, because the model learned which words the humans picked, not
#           what helpfulness is. That is reward hacking in one cell.
# ===============================================
REPEATS = 1          # <-- try 2, then 5

def reward(text):
    v = torch.tensor(vectorizer.transform([text]).toarray(), dtype=torch.float32)
    with torch.no_grad():
        return reward_model(v).item()

# Built from the words the model itself rewards, not from words we guessed.
best_words = [vocab[i] for i in order[-6:]]
gamed = " ".join(best_words * REPEATS)

real = "Open Settings, go to Security, and choose Reset Password."
print(f"   real answer  reward {reward(real):+6.2f}   {real}")
print(f"   gamed        reward {reward(gamed):+6.2f}   {gamed[:60]}")
print()
print("gaming wins:", reward(gamed) > reward(real))

## Answers

Try each task before reading.

In [ ]:
# YOUR TURN 1
#   Answers using the vocabulary of the preferred group score positive. Nothing
#   here understands passwords; it learned which words the humans kept choosing.
#
# YOUR TURN 2
#   The reward is linear in word counts, so repeating the six best words scales
#   the score without limit: +8.7, +17.3, +42.9 at 1, 2 and 5 repeats, against
#   +8.8 for a real answer. Two defences from lecture: a KL penalty keeps the
#   policy near the original model so it cannot drift into nonsense, and fresh
#   human comparisons let the reward model see the exploit and score it down.